# 模組 5：例外處理 (try/except)

目標：練習捕捉常見錯誤並提供友善訊息。

---

## Python 內建例外型態一覽

| 例外型態 | 觸發原因 |
|---|---|
| `ValueError` | 值的型態正確但內容不合法 |
| `TypeError` | 操作或函式使用了錯誤的型態 |
| `ZeroDivisionError` | 除以 0 |
| `IndexError` | 序列索引超出範圍 |
| `KeyError` | 字典中找不到指定的鍵 |
| `AttributeError` | 物件沒有該屬性或方法 |
| `NameError` | 變數名稱未定義 |
| `FileNotFoundError` | 找不到指定的檔案 |
| `ImportError` | 模組無法匯入 |
| `ModuleNotFoundError` | 找不到指定模組（ImportError 子類別） |
| `OverflowError` | 數值運算結果超出範圍 |
| `RecursionError` | 超過最大遞迴深度 |
| `StopIteration` | 迭代器已無更多元素 |
| `OSError` / `IOError` | 作業系統層級錯誤 |
| `PermissionError` | 沒有權限存取檔案或資源 |
| `TimeoutError` | 操作逾時 |
| `UnicodeDecodeError` | 解碼位元組為字串失敗 |
| `UnicodeEncodeError` | 編碼字串為位元組失敗 |
| `AssertionError` | assert 條件為 False |
| `RuntimeError` | 不屬於其他類別的執行時期錯誤 |
| `NotImplementedError` | 抽象方法尚未實作 |
| `MemoryError` | 記憶體不足 |

---
## 1. ValueError — 值的內容不合法

In [ ]:
raw = "abc"  # 試試改成 "10" 觀察正常流程
source
## 15. AssertionError — 斷言失敗

說明：當 `assert` 陳述式的條件為 False，或程式以 `raise AssertionError(...)` 顯式拋出時，Python 會產生 `AssertionError`。`assert` 常用於開發或測試期間檢查不變式（invariants）或前置條件，而不是用於對外部或使用者輸入的驗證。

範例：
```python
assert x > 0, 'x 必須為正數'
```

何時會發生：
- `assert` 條件評估為 False（例如：`assert len(lst) > 0` 但 `lst == []`）
- 程式碼中手動 `raise AssertionError(...)`

注意：
- 使用 `python -O`（optimize）執行時，所有 `assert` 陳述式會被移除，因而不應依賴 `assert` 來執行生產環境的必要檢查或邏輯。
- 對使用者輸入或外部資料進行驗證時，應使用顯式條件檢查並 `raise ValueError` 或其他明確例外，而非 `assert`。
- 單元測試中 `assert` 很方便，pytest 會改寫 `assert` 以提供更清楚的失敗資訊。

處理建議：
- 在需要的情況下用 `try/except AssertionError as e:` 捕捉並記錄錯誤（測試或工具中常見），但在一般程式流程中避免用例外控制正常邏輯。
- 提供具體的錯誤訊息以利除錯。

良好實務範例（輸入驗證）：
```python
def divide(a, b):
    if b == 0:
        raise ValueError("除數不能為 0")
    return a / b
```

避免 assert 中斷：設計建議

- 核心觀念：`assert` 用於開發/測試期間的內部檢查；以 `python -O` 執行時會被移除，因此不要把 `assert` 當作生產環境的輸入或流程檢查。
- 原則：對外部輸入做明確驗證並拋出具名例外（`ValueError`/`TypeError`等）；在需要容錯的地方使用 `try/except` 或回退邏輯；把 `assert` 留給測試或開發自檢。
- 實作建議：使用 guard clauses、型別檢查、資料模型 (dataclass / pydantic)、和單元測試來捕捉錯誤，而非用 `assert False` 中斷流程。

範例（多個短例子，示範如何用穩健方式取代 `assert False`）：

1) 明確輸入驗證並拋出具名例外
```python
def divide(a, b):
    if b == 0:
        raise ValueError("除數不能為 0")
    return a / b
```

2) 參數型別檢查（避免隱性斷言）
```python
def process(items):
    if not isinstance(items, list):
        raise TypeError("items 必須為 list")
    # 處理 items...
```

3) 類別初始化時檢查（dataclass 範例）
```python
from dataclasses import dataclass

@dataclass
class Account:
    balance: float
    def __post_init__(self):
        if self.balance < 0:
            raise ValueError("balance 必須 >= 0")
```

4) EAFP（直接嘗試，必要時捕捉 AttributeError）— 在可能缺少方法時用 try/except 處理
```python
def append_if_supported(obj, item):
    try:
        obj.append(item)
    except AttributeError:
        # fallback 或轉換
        return [*(obj if isinstance(obj, (list, tuple)) else [obj]), item]
    return obj
```

5) 對「不應發生但可能發生」的情況，拋出明確例外（取代 `assert False`）
```python
def handle_case(x):
    if x == 1:
        return "A"
    if x == 2:
        return "B"
    raise RuntimeError(f"Unknown case: {x}")  # 比 assert False 更清楚且可被捕捉
```

6) 使用驗證框架（如 `pydantic`）做資料驗證（更適合複雜輸入）
```python
from pydantic import BaseModel, ValidationError

class Item(BaseModel):
    name: str
    qty: int

# Item(name='pen', qty=-1)  # 將拋出 ValidationError
```

7) 在測試中使用 `assert`（測試框架負責檢查；生產程式用具名例外）
```python
# tests/test_divide.py (pytest)
def test_divide_by_zero():
    with pytest.raises(ValueError):
        divide(1, 0)
```

補充小提醒：
- 若程式中仍需短暫「內部檢查」，可用 `assert`，但千萬別依賴它作為輸入驗證或商業邏輯判斷。
- 若要保留開發期檢查且在生產提供不同行為，可寫顯式條件並在開發模式下記 log 或 raise。
- 若不可避免需捕捉 `AssertionError`，把它轉換成更具意義的例外或記錄詳細資訊再處理。

[ValueError] 請輸入可轉為整數的內容：invalid literal for int() with base 10: 'abc'


---
## 2. TypeError — 型態不符

In [9]:
try:
    result = "數字" + 100  # 字串不能直接加整數
    print(result)
except TypeError as e:
    print(f"[TypeError] 型態不符：{e}")

[TypeError] 型態不符：can only concatenate str (not "int") to str


In [33]:
# Exercise
# 要 int 給 string

# 情境練習, 人口與排汙量的模型, 輸入x人口的數,得到y城鎮(排汙量/月)

a=10
b=20
x=str(input("Please input the population x of your city :"))

try:
    y:int=(a*x)+b
    print(f"Calculating ....., x:{x}, y:{y}")
except TypeError as te:
    print(f"[ Type error ] : {te}")
else :
    print(f"The pollution of your city is : {y}")
finally:
    print("Caculating complete.")




[ Type error ] : can only concatenate str (not "int") to str
Caculating complete.


---
## 3. ZeroDivisionError — 除以 0

In [26]:
try:
    result = 100 / 0
    print("結果：", result)
except ZeroDivisionError as e:
    print(f"[ZeroDivisionError] 不可除以 0：{e}")

[ZeroDivisionError] 不可除以 0：division by zero


In [ ]:
# Exercise 
# 計算機UI 底層程式設計

def division(a:int,b:int) -> float :
    return a/b

x=int(input("Please input mumerator A :"))
y=int(input("Please input denominator B :"))

try : 
    print(f"Result of a / b is {division(x,y)}")
except ZeroDivisionError as zde :
    print(f"[ZeroDivisionError] err msg : {zde} ")
else :
    print("done....")

[ZeroDivisionError] err msg :division by zero


---
## 4. IndexError — 索引超出範圍

In [ ]:
fruits = ["apple", "banana", "cherry"]

try:
    print(fruits[10])  # 只有 index 0~2
except IndexError as e:
    print(f"[IndexError] 索引超出範圍：{e}")

In [57]:
# Exercise
# 

THSR_stations="Nangang,Taipei,Banqiao,Taoyua,Hsinchu,Maioli,Taichung,Changhua,Yunlin,Chiayi,Tainan,Zuoying".split(",")
# print(THSR_staion)

index_THSR=int(input(f"There are toal {len(THSR_stations)}, whicj index you wanna serach ? :"))

try:
    if index_THSR <= 0 :
        raise IndexError("Please input an index that 'non-negative'.")
    print(f"The station of number {int(index_THSR)} is {THSR_stations[index_THSR-1]}")
except IndexError as ie:
    print(f"{ie}\nPlease Search the index between [1, {int(len(THSR_station))}]")
except ValueError as ve:
    print(f"{ve}\nPlease input integer again, thanks.")


Please input an index that 'non-negative'.
Please Search the index between [1, 12]


---
## 5. KeyError — 字典找不到鍵

In [66]:
person = {"name": "Alice", "age": 30}

try:
    print(person["email"])  # 'email' 鍵不存在
except KeyError as e:
    print(f"[KeyError] 找不到鍵：{e}")

[KeyError] 找不到鍵：'email'


In [ ]:
uut_list = [
    {"SN":"xxxx1","brand":"Pixel","Market Name":"Pixel 10 Pro XL"},
    {"SN":"xxxx2","brand":"Samsung","Market Name":"Samsung S26 Ultra"},
    {"SN":"xxxx3","brand":"Apple","Market Name":"IPhone 17 Pro Max"} 
]


# print(f"{uut_list[0]["Manufacture"]}")

try:
    print(f"{uut_list[0]["Manufacture"]}")
except KeyError as ke :
    print(f"Not found any Key : {ke}")


KeyError: 'Manufacture'

---
## 6. AttributeError — 物件沒有該屬性或方法

說明：當程式試圖存取或呼叫一個物件不存在的屬性或方法時，Python 會引發 `AttributeError`。這通常代表變數的型別或狀態與程式預期不符。

常見原因：
- 變數型態不如預期（例如把 `list` 當成 `dict` 使用）
- 呼叫不存在的方法（例如對 `int` 使用 `.append()`）
- 屬性拼寫錯誤或物件尚未初始化

處理建議：
- 在呼叫前使用 `hasattr(obj, 'attr')` 或檢查型態以避免錯誤。
- 在必要時用 `try/except AttributeError` 捕捉例外並提供友善的錯誤訊息或替代邏輯。
- 改善設計（明確型別或初始化流程），減少在正常流程中依賴例外控制流程。

簡短範例：
```python
if hasattr(obj, 'append'):
    obj.append(item)
else:
    raise AttributeError('obj 不支援 append()')
```

---
**Dict（字典）常用方法（部分）**
- `dict.get(key[, default])`：取得鍵對應值，鍵不存在時回傳 `default`（避免 KeyError）。
- `dict.keys()`：回傳鍵的視圖 (view)。
- `dict.values()`：回傳值的視圖。
- `dict.items()`：回傳 (key, value) 對的視圖。
- `dict.update(other)`：用另一個映射或鍵值對更新字典。
- `dict.pop(key[, default])`：移除鍵並回傳其值；若不存在且未給 default，會引發 KeyError。
- `dict.setdefault(key[, default])`：若鍵不存在則設定預設值並回傳該值。
- `dict.clear()`：清空字典。
- `dict.copy()`：淺複製字典。
- `dict.popitem()`：移除並回傳任一 (key, value)（Python 3.7 起為 LIFO）。
- `dict.fromkeys(iterable, value=None)`：從可迭代物建新字典。

**List（清單）常用方法（部分）**
- `list.append(x)`：在尾端新增元素。
- `list.extend(iterable)`：延伸清單，加入多個元素。
- `list.insert(i, x)`：在指定位置插入元素。
- `list.remove(x)`：移除第一個符合的值，否則 ValueError。
- `list.pop([i])`：移除並回傳指定索引的元素，預設為最後一個。
- `list.clear()`：清空清單。
- `list.index(x[, start[, end]])`：回傳第一個符合值的索引。
- `list.count(x)`：回傳值在清單中出現次數。
- `list.sort(key=None, reverse=False)`：就地排序。
- `list.reverse()`：就地反轉。
- `list.copy()`：淺複製清單。

簡短示例：
```python
# dict.get 範例
data = {'a': 1}
print(data.get('b', 0))  # 0 (不會 KeyError)

# list.append 範例
arr = [1, 2]
arr.append(3)
print(arr)  # [1, 2, 3]
```

In [ ]:
try:
    number = 42
    number.append(1)  # int 沒有 append() 方法
except AttributeError as e:
 
    print(f"[AttributeError] 屬性不存在：{e}")


In [5]:
# list 才有 append() 方法，int 沒有，所以會引發 AttributeError。
# Exercise

# 這邊改用其他做法. 對 list 使用 字典的 method
# dictionary 的 Method :
# dict.get() , dict.keys() , dict.values() ,
# dict.items() , dict.update() , dict.pop() , 
# dict.clear() , dict.copy() , dict.fromkeys() 
# dict.setdefault() , dict.popitem() ,
# dict.viewkeys() , dict.viewvalues() , dict.viewitems()

# list_a=["xxxx1","Pixel","Pixel 10 Pro XL"]
# print(f"Original dict : {dict_a}")
# print(f"Try to use dict method on list : {dict_a.get('SN')}")


try:
    list_a=["xxxx1","Pixel","Pixel 10 Pro XL"]
    print(f"Try to use dict method on list : {list_a.get('SN')}")
except AttributeError as ae:
    print(f"Not found any method : {ae}")

Not found any method : 'list' object has no attribute 'get'


---
## 7. NameError — 變數未定義

In [ ]:
try:
    print(undefined_variable)  # 此變數從未被宣告
except NameError as e:
    print(f"[NameError] 變數未定義：{e}")

---
## 8. FileNotFoundError — 找不到檔案

In [ ]:
try:
    with open("ghost_file.txt", "r") as f:
        content = f.read()
except FileNotFoundError as e:
    print(f"[FileNotFoundError] 找不到檔案：{e}")

---
## 9. ModuleNotFoundError — 找不到模組

In [ ]:
try:
    import non_existent_module
except ModuleNotFoundError as e:
    print(f"[ModuleNotFoundError] 找不到模組：{e}")

---
## 10. OverflowError — 數值超出範圍

In [ ]:
import math

try:
    result = math.exp(100000)  # e^100000 超出 float 可表示的範圍
    print(result)
except OverflowError as e:
    print(f"[OverflowError] 數值超出範圍：{e}")

---
## 11. RecursionError — 超過最大遞迴深度

In [ ]:
def infinite_recursion():
    return infinite_recursion()  # 無限遞迴，沒有終止條件

try:
    infinite_recursion()
except RecursionError as e:
    print(f"[RecursionError] 超過最大遞迴深度：{e}")

---
## 12. StopIteration — 迭代器耗盡

In [ ]:
my_iter = iter([1, 2])  # 只有 2 個元素

try:
    print(next(my_iter))  # 取第 1 個
    print(next(my_iter))  # 取第 2 個
    print(next(my_iter))  # 已無元素 → StopIteration
except StopIteration:
    print("[StopIteration] 迭代器已無更多元素")

---
## 13. UnicodeDecodeError — 解碼失敗

In [ ]:
try:
    b = b"\xff\xfe"  # 無效的 UTF-8 位元組序列
    text = b.decode("utf-8")
except UnicodeDecodeError as e:
    print(f"[UnicodeDecodeError] 解碼失敗：{e}")

---
## 14. UnicodeEncodeError — 編碼失敗

In [ ]:
try:
    text = "你好"  # 中文字
    encoded = text.encode("ascii")  # ASCII 無法編碼中文
except UnicodeEncodeError as e:
    print(f"[UnicodeEncodeError] 編碼失敗：{e}")

---
## 15. AssertionError — 斷言失敗

說明：當 `assert` 陳述式的條件為 False，或程式以 `raise AssertionError(...)` 顯式拋出時，Python 會產生 `AssertionError`。`assert` 常用於開發或測試期間檢查不變式（invariants）或前置條件，而不是用於對外部或使用者輸入的驗證。

範例：
```python
assert x > 0, 'x 必須為正數'
```

何時會發生：
- `assert` 條件評估為 False（例如：`assert len(lst) > 0` 但 `lst == []`）
- 程式碼中手動 `raise AssertionError(...)`

注意：
- 使用 `python -O`（optimize）執行時，所有 `assert` 陳述式會被移除，因而不應依賴 `assert` 來執行生產環境的必要檢查或邏輯。
- 對使用者輸入或外部資料進行驗證時，應使用顯式條件檢查並 `raise ValueError` 或其他明確例外，而非 `assert`。
- 單元測試中 `assert` 很方便，pytest 會改寫 `assert` 以提供更清楚的失敗資訊。

處理建議：
- 在需要的情況下用 `try/except AssertionError as e:` 捕捉並記錄錯誤（測試或工具中常見），但在一般程式流程中避免用例外控制正常邏輯。
- 提供具體的錯誤訊息以利除錯。

良好實務範例（輸入驗證）：
```python
def divide(a, b):
    if b == 0:
        raise ValueError('除數不能為 0')
    return a / b
```

捕捉 AssertionError 的範例：
```python
try:
    assert False, '測試失敗'
except AssertionError as e:
    print('AssertionError:', e)
```

In [ ]:
# 穩健替代寫法示範：safe_divide / try_int / warnings / logging / 轉換 AssertionError
def safe_divide(a, b):
    """生產環境使用：若 b == 0，拋出 ValueError"""
    if b == 0:
        raise ValueError("除數不能為 0")
    return a / b

def try_int(s):
    try:
        return int(s), None
    except ValueError as e:
        return None, str(e)

import warnings
def process_scores(scores):
    if not scores:
        warnings.warn('scores 為空，回傳空列表', RuntimeWarning)
        return []
    total = sum(scores)
    return [s / total for s in scores]

import logging
logger = logging.getLogger(__name__)
DEFAULT_CFG = {'mode': 'safe'}
def load_config(cfg):
    if not isinstance(cfg, dict) or 'mode' not in cfg:
        logger.error('config 無效，使用預設值')
        return DEFAULT_CFG.copy()
    return cfg

# 捕捉並轉換 AssertionError 的示範
def check_and_convert(cond):
    try:
        assert cond(), '條件不成立'
    except AssertionError as e:
        raise ValueError('轉換後的錯誤訊息') from e

# 範例運行（示範）
print('safe_divide(10,2) ->', safe_divide(10,2))
print("try_int('10') ->", try_int('10'))
print("try_int('x') ->", try_int('x'))
print('process_scores([]) ->', process_scores([]))
print('load_config({}) ->', load_config({}))
try:
    check_and_convert(lambda: False)
except ValueError as e:
    print('check_and_convert raised:', e)

[AssertionError] 斷言失敗：除數不能為 0


---
## 16. NotImplementedError — 方法尚未實作

In [ ]:
class Animal:
    def speak(self):
        raise NotImplementedError("子類別必須實作 speak() 方法")

class Dog(Animal):
    pass  # 忘記實作 speak()

try:
    d = Dog()
    d.speak()
except NotImplementedError as e:
    print(f"[NotImplementedError] {e}")

---
## 17. 多重例外 + else + finally

In [ ]:
# 試著修改 raw 的值，觀察不同的例外路徑
raw = "5"   # 試試 "0", "abc", "5"

try:
    value = int(raw)
    result = 100 / value
except ValueError:
    print("請輸入可轉為整數的內容")
except ZeroDivisionError:
    print("不可除以 0")
else:
    # 只有在沒有發生例外時才執行
    print("result:", result)
finally:
    # 無論是否發生例外都會執行
    print("--- 執行完畢 ---")

---
## 18. 捕捉所有例外 (Exception)

> ⚠️ 通常只在除錯時使用，生產環境建議明確指定例外型態。

In [5]:
try:
    data = {"key": "value"}
    print(data["missing_key"])   # KeyError
except Exception as e:
    print(f"發生了例外：{type(e).__name__} → {e}")

發生了例外：KeyError → 'missing_key'


In [ ]:
try:
    input=()
except Exception as come_from_origin:
    print(f"Error msg comes from origin :{come_from_origin}")
else:
    print(f"Everything goes well.")
finally:
    print("END")

SyntaxError: invalid syntax (3571444198.py, line 2)